# Orbit Wars — Colab Training Pipeline

GPU-accelerated RL training on Google Colab with persistent storage via Google Drive.

**Workflow:**
1. Run the Setup cell (installs deps, mounts Drive, clones repo)
2. Edit the Config cell to choose your experiment
3. Run Training
4. Monitor with TensorBoard, evaluate, generate submission

All checkpoints and logs are saved to Google Drive so they persist across sessions.

In [ ]:
#@title 1. Setup: Mount Drive, Clone Repo, Install Dependencies

import os, sys

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Persistent output directory on Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/orbit_wars_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/submissions', exist_ok=True)
print(f'Drive output dir: {DRIVE_OUTPUT}')

# ── Clone or update repo ────────────────────────────────────────────────────
REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT THIS
REPO_DIR = '/content/orbit_wars'

if os.path.exists(REPO_DIR):
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')

# ── Install dependencies ────────────────────────────────────────────────────
os.system('pip install -q --upgrade "kaggle-environments>=1.28.0"')
os.system('pip install -q "stable-baselines3[extra]>=2.3" gymnasium pyyaml')

print('\nSetup complete.')

In [ ]:
#@title 2. GPU Verification

import torch

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU detected! Training will be slow.')
    print('Go to Runtime > Change runtime type > GPU')
    DEVICE = 'cpu'

print(f'\nUsing device: {DEVICE}')

## Configuration

Edit the cell below to configure your experiment. The transformer PPO config is the default.
Override values as needed (e.g., change `total_updates`, `device`, `opponent`).

In [ ]:
#@title 3. Experiment Config

from src.config import load_train_config, train_config_from_dict

# Load from YAML config
cfg = load_train_config('configs/transformer_ppo.yaml')

# Override for Colab GPU training
cfg.device = 'cuda'
cfg.ppo.num_envs = 4
cfg.ppo.total_updates = 2000

# Point outputs to Google Drive for persistence
cfg.save_dir = f'{DRIVE_OUTPUT}/checkpoints'
cfg.log_dir = f'{DRIVE_OUTPUT}/logs'

print(f'Run: {cfg.run_name}')
print(f'Device: {cfg.device}')
print(f'Updates: {cfg.ppo.total_updates}')
print(f'Envs: {cfg.ppo.num_envs}')
print(f'Opponent: {cfg.opponent}')

In [ ]:
#@title 4. Train

import yaml, sys, os
sys.path.insert(0, '/content/orbit_wars')
os.chdir('/content/orbit_wars')

# Write config to temp file for the training script
temp_cfg = '/tmp/train_cfg.yaml'
with open(temp_cfg, 'w') as f:
    yaml.dump({
        'seed': cfg.seed,
        'run_name': cfg.run_name,
        'device': cfg.device,
        'save_dir': cfg.save_dir,
        'log_dir': cfg.log_dir,
        'checkpoint_every': cfg.checkpoint_every,
        'log_every': cfg.log_every,
        'opponent': cfg.opponent,
        'alternate_player_sides': cfg.alternate_player_sides,
        'env': {
            'max_targets': cfg.env.max_targets,
            'k_neighbors': cfg.env.k_neighbors,
            'ship_fractions': cfg.env.ship_fractions,
        },
        'model': {
            'embed_dim': cfg.model.embed_dim,
            'n_heads': cfg.model.n_heads,
            'n_layers': cfg.model.n_layers,
            'ff_dim': cfg.model.ff_dim,
            'pos_hidden': cfg.model.pos_hidden,
        },
        'ppo': {
            'rollout_steps': cfg.ppo.rollout_steps,
            'num_envs': cfg.ppo.num_envs,
            'total_updates': cfg.ppo.total_updates,
            'epochs': cfg.ppo.epochs,
            'minibatch_size': cfg.ppo.minibatch_size,
            'gamma': cfg.ppo.gamma,
            'clip_coef': cfg.ppo.clip_coef,
            'ent_coef': cfg.ppo.ent_coef,
            'vf_coef': cfg.ppo.vf_coef,
            'lr': cfg.ppo.lr,
            'max_grad_norm': cfg.ppo.max_grad_norm,
        },
        'reward': {
            'sparse': cfg.reward.sparse,
            'dense_ship_coef': cfg.reward.dense_ship_coef,
            'dense_prod_coef': cfg.reward.dense_prod_coef,
        },
    }, f)

!python -m src.train --config {temp_cfg}
print(f'\nCheckpoints saved to: {cfg.save_dir}/{cfg.run_name}')

## Experiment Sweep

Run multiple experiments with different hyperparameters. Results are saved to separate subdirectories on Drive.

In [ ]:
#@title 5. Hyperparameter Sweep (optional)

from itertools import product
from training.train import load_config, apply_dotted_overrides, train

# Define sweep parameters
sweep = {
    'training.learning_rate': [3e-4, 1e-4],
    'training.ent_coef': [0.01, 0.005],
}

# Reduce timesteps per experiment for faster sweep
SWEEP_TIMESTEPS = 200_000

keys = list(sweep.keys())
values = list(sweep.values())
results = []

for combo in product(*values):
    overrides = [f'{k}={v}' for k, v in zip(keys, combo)]
    run_label = '_'.join(f'{k.split(".")[-1]}={v}' for k, v in zip(keys, combo))
    print(f'\n{"="*60}\nSweep: {run_label}\n{"="*60}')

    cfg = load_config('configs/ppo_default.yaml')
    cfg = apply_dotted_overrides(cfg, [
        f'training.total_timesteps={SWEEP_TIMESTEPS}',
        'training.device=cuda',
        'env.n_envs=8',
        f'output.run_name=sweep_{run_label}',
    ] + overrides)
    cfg.setdefault('output', {})
    cfg['output']['checkpoint_dir'] = f'{DRIVE_OUTPUT}/checkpoints'
    cfg['output']['log_dir'] = f'{DRIVE_OUTPUT}/logs'

    _, ckpt_dir = train(cfg)
    results.append((run_label, ckpt_dir))

print(f'\n\nSweep complete! {len(results)} experiments ran.')
for label, path in results:
    print(f'  {label} -> {path}')

## Monitoring

In [ ]:
#@title 6. TensorBoard

%load_ext tensorboard
%tensorboard --logdir {DRIVE_OUTPUT}/logs

## Evaluation

In [ ]:
#@title 7. Evaluate Trained Model

from pathlib import Path
from agents.rl_agent import RLAgent
from evaluation.evaluate import benchmark

# Use the latest training run, or specify a path:
model_path = checkpoint_dir / 'best_model.zip'
# model_path = Path(f'{DRIVE_OUTPUT}/checkpoints/ppo_colab_XXXXXXXX_XXXXXX/best_model.zip')

if model_path.exists():
    rl_agent = RLAgent(model_path)
    benchmark(rl_agent, agent_name='rl_colab', n_games=20)
else:
    print(f'Model not found: {model_path}')
    print('Available checkpoints on Drive:')
    for p in sorted(Path(f'{DRIVE_OUTPUT}/checkpoints').glob('*/best_model.zip')):
        print(f'  {p}')

## Submission

In [ ]:
#@title 8. Generate Submission

from agents.rl_agent import export_submission

submission_path = f'{DRIVE_OUTPUT}/submissions/submission_rl.py'

if model_path.exists():
    export_submission(model_path, submission_path, mode='rl')
    print(f'Submission saved to: {submission_path}')
    print(f'\nDownload from Google Drive or upload directly:')
    print(f'  kaggle competitions submit orbit-wars -f "{submission_path}" -m "Colab PPO"')
else:
    print('No model to export. Train first (cell 4).')